우리는 지금까지 **코사인 유사도**를 배웠습니다. 코사인 유사도는 단어가 몇 번 나왔는지(빈도)와 단어의 가중치(TF-IDF)를 중요하게 여깁니다. 하지만 다음과 같은 상황에서는 '단순히 어떤 요소들이 들어있는가'를 보는 집합 기반 방식이 더 유리합니다.

1.  **구성 요소가 명확할 때:** 문서의 키워드 목록, 태그 집합 등을 비교할 때.
2.  **문서의 길이가 크게 다를 때:** 매우 긴 문서와 짧은 문서를 비교하면 코사인 유사도는 왜곡될 수 있지만, 집합 기반은 '공통 분모'에 집중합니다.
3.  **오타가 섞여 있을 때:** 단어 단위가 아닌 글자 단위 집합(n-그램)으로 쪼개서 비교하면 오타가 있어도 유사도를 찾아낼 수 있습니다.

## 3. 핵심 용어 및 개념 정리

### 3.1 자카드 유사도 (Jaccard Similarity)
가장 대표적인 집합 유사도입니다. **"전체 등장한 것들 중 공통된 것이 얼마나 되는가?"** 를 측정합니다.
*   **핵심:** 합집합 대비 교집합의 비율 (0~1)

### 3.2 다이스 유사도 (Dice Coefficient)
자카드와 비슷하지만, **"두 집합의 공통된 부분"에 더 많은 가중치** 를 줍니다.
*   **특징:** 분모에서 전체 합집합을 구하는 대신 각 집합의 크기를 단순히 더하기 때문에, 자카드보다 유사도 수치가 높게 나오는 경향이 있습니다.

### 3.3 셈블 (Shingle) 또는 n-그램
문장을 아주 작은 조각으로 나누는 기법입니다. '지붕의 기와(Shingle)'처럼 단어나 글자를 겹쳐서 나누는 모습에서 유래했습니다.
*   **문자 2-그램:** "apple" → {'ap', 'pp', 'pl', 'le'}
*   **용도:** 단어가 조금 틀려도 조각들이 일치하면 유사하다고 판단할 수 있게 해줍니다.

### 3.4 MinHash (민해시)
데이터가 너무 많아서 일일이 집합을 비교할 수 없을 때 쓰는 **'요약 기술'** 입니다. 
*   **비유:** 두 사람의 성격이 비슷한지 보려고 모든 과거 기록을 대조하는 대신, 몇 가지 '질문(해시 함수)'을 던져서 그 답변(서명)이 일치하는지 보는 것과 같습니다.

## 4. 예제로 배우는 자카드 vs 다이스

두 문서의 키워드 집합을 비교해 봅시다.

*   **문서 A:** {자연어, 처리, 머신, 러닝} (크기: 4)
*   **문서 B:** {머신, 러닝, 딥러닝, 인공지능} (크기: 4)

### Step 1: 기본 연산
1.  **교집합 (공통 단어):** {머신, 러닝} → **2개**
2.  **합집합 (모든 단어):** {자연어, 처리, 머신, 러닝, 딥러닝, 인공지능} → **6개**

### Step 2: 자카드 계산
$$J(A, B) = \frac{교집합}{합집합} = \frac{2}{6} \approx 0.33$$
(전체 6개 단어 중 33%가 공통임)

### Step 3: 다이스 계산
$$\text{Dice}(A, B) = \frac{2 \times 교집합}{|A| + |B|} = \frac{2 \times 2}{4 + 4} = \frac{4}{8} = 0.5$$
(공통 부분에 2배 가중치를 주어 0.5가 나옴)

## 5. 오타를 잡아내는 셈블(n-그램)의 마법

만약 "apple"과 "aple"(오타)를 단어 자체로 비교하면 유사도는 0입니다. 하지만 **문자 2-그램(Shingle)** 으로 쪼개면 어떨까요?

*   **Set A (apple):** {ap, pp, pl, le}
*   **Set B (aple):** {ap, pl, le}

1.  **교집합:** {ap, pl, le} (3개)
2.  **합집합:** {ap, pp, pl, le} (4개)
3.  **자카드 유사도:** $3/4 = 0.75$

**결론:** 단어는 다르지만 75%나 유사하다고 나옵니다! 이것이 검색 엔진이나 맞춤법 검사기에서 유사한 단어를 찾는 원리입니다.

## 6. 대규모 데이터의 해결사: MinHash

만약 웹 페이지 1,000만 개 중에서 서로 비슷한 페이지를 찾으려면 어떻게 할까요? 모든 페이지 쌍을 비교하려면 수십 조 번의 계산이 필요합니다. 이때 **MinHash**가 등장합니다.

### 6.1 핵심 아이디어: "서명(Signature) 만들기"
전체 텍스트를 다 비교하지 말고, 각 문서를 대표하는 **짧은 암호(서명)** 를 만듭니다.

1.  수백 개의 **해시 함수(일종의 무작위 순서 정렬기)** 를 준비합니다.
2.  각 함수마다 집합의 원소들을 마구 섞은 후, 가장 앞에 오는(최솟값) 원소를 하나씩 뽑습니다.
3.  이 뽑힌 값들의 리스트가 바로 **'서명'** 입니다.

### 6.2 왜 이게 자카드와 같나요?
수학적으로 **"두 문서의 서명 값이 우연히 일치할 확률"은 "두 문서의 실제 자카드 유사도"와 같습니다.** 

*   따라서 실제 집합을 다 뒤질 필요 없이, 짧은 서명 리스트만 대조해서 "100개 중 80개가 일치하네? 그럼 이 두 문서는 80% 정도 비슷하겠군!"이라고 **근사치** 를 빠르게 계산할 수 있습니다.